# Predicting "Choose your track"

We train four classifiers on the survey answers in `Responses.csv` to predict the track each student chose:

1. Logistic Regression
2. Support Vector Machine (SVM)
3. Random Forest
4. K-Nearest Neighbors (KNN)

The notebook covers:

- **Data quality:** remove rows with random answers (the biggest accuracy gain) and duplicate rows (so the test scores are honest).
- **Training:** tune each model's hyperparameters with 5-fold cross-validation on the training set.
- **Evaluation:** confusion matrix, accuracy, precision, recall and F1-score on a held-out test set, plus cross-validation accuracy to check that the scores are stable and the models do not overfit.
- **Analysis:** where the remaining errors come from and which questions matter most.

In [1]:
import re

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pd.set_option("display.max_colwidth", None)

RANDOM_STATE = 42
TARGET = "Choose your track"
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 1. Load the data

In [2]:
df = pd.read_csv("Responses.csv")
print("Shape:", df.shape)
df.head()

Shape: (8416, 20)


,Unnamed: 0,Choose your track,1- What is your current level expertise in computer programming?,2- How often do you code or work on tech related projects in your free time?,3- How often do you keep up with tech news and trends?,4- Were you willing to learn and work with programming languages?,"5- Was your choice based on enjoying working with visual elements, such as layout and design?","7- Did you have interest in analyzing and studying user's behavior , needs and challenges?",8- Did you have a strong background in mathematics and enjoyed solving complex mathematical problems?,9-Did you enjoy working with data to solve real world problems?,10- Were you interested in data research and analysis?,11- Were you passionate about data analysis and deriving insights from data sets?,12- Did you have interest in using algorithms that can learn and improve from data?,13- Were you interested in server-side programming and building the backbone of applications?,14- Did you have interest in creating innovative and creative software?,15- Was you choice based on using mobile apps frequently and having ideas for new ones?,16- Were you concerned about the security of digital systems and data?,"17- Did you enjoy learning about encryption, firewalls and network protection",18- Did you enjoy creative problem-solving and design thinking?,6- Did you have interest creating responsive and interactive web pages
0,NaN,Front-end,Intermediate,Weekly,Weekly,Yes,Yes,yes,No,No,No,No,No,No,Yes,No,No,No,Yes,Yes
1,NaN,Front-end,Beginner,Weekly,Weekly,Yes,Yes,No,Yes,Yes,No,No,Yes,Yes,Yes,No,Yes,No,Yes,Yes
2,NaN,Back-end,Expert,Daily,Weekly,Yes,No,No,No,No,Yes,No,No,Yes,Yes,No,Yes,Yes,No,Yes
3,NaN,UI/UX,Intermediate,Weekly,Daily,No,Yes,yes,No,No,No,No,No,No,No,No,No,No,No,No
4,NaN,Data Science,Beginner,Daily,Daily,Yes,No,No,Yes,Yes,Yes,Yes,No,No,No,No,No,No,Yes,No


## 2. Clean the data

The first column has no header and is completely empty, so we drop it.

The question headers are long, so we rename them to `Q1` … `Q18` using the number at the start of each question. The CSV lists question 6 last, so we also put the columns back in order.

In [3]:
df = df.drop(columns=["Unnamed: 0"])

short_names = {col: "Q" + re.match(r"\d+", col).group() for col in df.columns if col != TARGET}
question_cols = sorted(short_names.values(), key=lambda q: int(q[1:]))
df = df.rename(columns=short_names)[[TARGET] + question_cols]

# Reference table: short name -> full question
question_text = pd.Series({short: full for full, short in short_names.items()}, name="Question")[question_cols]
question_text.to_frame()

,Question
Q1,1- What is your current level expertise in computer programming?
Q2,2- How often do you code or work on tech related projects in your free time?
Q3,3- How often do you keep up with tech news and trends?
Q4,4- Were you willing to learn and work with programming languages?
Q5,"5- Was your choice based on enjoying working with visual elements, such as layout and design?"
Q6,6- Did you have interest creating responsive and interactive web pages
Q7,"7- Did you have interest in analyzing and studying user's behavior , needs and challenges?"
Q8,8- Did you have a strong background in mathematics and enjoyed solving complex mathematical problems?
Q9,9-Did you enjoy working with data to solve real world problems?
Q10,10- Were you interested in data research and analysis?


In [4]:
print("Missing values:", df.isna().sum().sum())
df[TARGET].value_counts()

Missing values: 0


Choose your track
Back-end               1267
UI/UX                  1222
Front-end              1208
Mobile applications    1196
Cyber Security         1185
Data Science           1178
Machine Learning       1160
Name: count, dtype: int64

## 3. Encode the answers as numbers

The models need numeric inputs. Every answer has a natural order, so we map each one to an integer:

- **Q1** (programming level): Beginner = 0, Intermediate = 1, Expert = 2
- **Q2, Q3** (how often): Never = 0, Rarely = 1, Monthly = 2, Weekly = 3, Daily = 4
- **Q4–Q18** (Yes/No): No = 0, Yes = 1. Q7 is written as lowercase `yes`, so we lowercase all answers before mapping.

In [5]:
level = {"Beginner": 0, "Intermediate": 1, "Expert": 2}
frequency = {"Never": 0, "Rarely": 1, "Monthly": 2, "Weekly": 3, "Daily": 4}
yes_no = {"no": 0, "yes": 1}

df["Q1"] = df["Q1"].map(level)
df["Q2"] = df["Q2"].map(frequency)
df["Q3"] = df["Q3"].map(frequency)
for col in question_cols[3:]:
    df[col] = df[col].str.strip().str.lower().map(yes_no)

# An answer that is not in the dictionaries above would turn into NaN.
assert df[question_cols].notna().all().all(), "Found an answer that is not in the mappings"
df.head()

,Choose your track,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18
0,Front-end,1,3,3,1,1,1,1,0,0,0,0,0,0,1,0,0,0,1
1,Front-end,0,3,3,1,1,1,0,1,1,0,0,1,1,1,0,1,0,1
2,Back-end,2,4,3,1,0,1,0,0,0,1,0,0,1,1,0,1,1,0
3,UI/UX,1,3,4,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
4,Data Science,0,4,4,1,0,0,0,1,1,1,1,0,0,0,0,0,0,1


## 4. Remove rows with random answers

Two blocks of rows do not look like real survey answers: **rows 5,554–6,039 and 7,870–8,415**, 1,032 rows in total. Row numbers are the DataFrame index, which starts at 0; add 2 to get the row number in Excel. The blocks were found by checking how well each row's answers fit its track. The next two cells show the evidence.

**Evidence 1: every question looks like a coin flip.** Inside the blocks, every Yes/No question is answered "Yes" about half the time. In the rest of the file the share of "Yes" changes a lot from question to question.

In [6]:
RANDOM_BLOCKS = [(5554, 6039), (7870, 8415)]

in_blocks = np.zeros(len(df), dtype=bool)
for start, end in RANDOM_BLOCKS:
    in_blocks |= (df.index >= start) & (df.index <= end)
print("Rows in the blocks:", in_blocks.sum())

# Share of "Yes" answers per question
pd.DataFrame({
    "Rest of the file": df.loc[~in_blocks, question_cols[3:]].mean(),
    "Random blocks": df.loc[in_blocks, question_cols[3:]].mean(),
}).T.round(2)

Rows in the blocks: 1032


,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18
Rest of the file,0.85,0.35,0.42,0.39,0.37,0.29,0.14,0.28,0.16,0.33,0.33,0.19,0.18,0.25,0.56
Random blocks,0.49,0.51,0.53,0.50,0.48,0.53,0.50,0.49,0.22,0.47,0.49,0.50,0.51,0.48,0.51


**Evidence 2: the answers say nothing about the track.** Cross-validation gives every unique row a prediction from a model that never saw that row. Before and between the blocks the predictions are mostly correct. Inside the blocks they are right about as often as guessing one of the 7 tracks at random (1/7 ≈ 14%).

In [7]:
unique_rows = df.drop_duplicates()
predicted = cross_val_predict(
    LogisticRegression(max_iter=3000), unique_rows[question_cols], unique_rows[TARGET], cv=cv
)
correct = pd.Series(predicted == unique_rows[TARGET].values, index=unique_rows.index)

windows = {
    "Before block 1 (rows 5,254-5,553)": (5254, 5553),
    "Block 1 (rows 5,554-6,039)": (5554, 6039),
    "Between the blocks (rows 6,040-7,869)": (6040, 7869),
    "Block 2 (rows 7,870-8,415)": (7870, 8415),
}
pd.DataFrame(
    [
        {"Part of the file": name, "Unique rows": correct.loc[start:end].size, "Accuracy": correct.loc[start:end].mean()}
        for name, (start, end) in windows.items()
    ]
).set_index("Part of the file").round(3)

,Unique rows,Accuracy
Part of the file,,
"Before block 1 (rows 5,254-5,553)",257,0.961
"Block 1 (rows 5,554-6,039)",486,0.134
"Between the blocks (rows 6,040-7,869)",258,0.938
"Block 2 (rows 7,870-8,415)",546,0.136


No model can predict a track from answers that carry no information about it. These rows only add noise to training and pull every test score down, so we remove them.

In [8]:
df = df[~in_blocks]
print("Rows left:", len(df))

Rows left: 7384


## 5. Remove duplicate rows

Most of the remaining rows are exact copies of another row (same answers and same track), and some responses appear up to 30 times. If we keep the copies, the same response ends up in both the training set and the test set. The models are then graded on answers they have already seen, and the test scores come out higher than they would be on new responses.

We keep one copy of each response. The index keeps the original row numbers, so we can still tell where each row came from in the file.

In [9]:
print("Rows before:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Most copies of a single response:", df.value_counts().max())

df = df.drop_duplicates()
print("Rows after:", len(df))
df[TARGET].value_counts()

Rows before: 7384
Duplicate rows: 5089
Most copies of a single response: 30
Rows after: 2295


Choose your track
Back-end               376
Mobile applications    349
UI/UX                  331
Machine Learning       316
Data Science           313
Front-end              305
Cyber Security         305
Name: count, dtype: int64

## 6. Split into training and test sets

80% of the rows are used for training and 20% are held back for testing. `stratify=y` keeps the same share of each track in both sets.

In [10]:
X = df[question_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape)
print("Test: ", X_test.shape)

Train: (1836, 18)
Test:  (459, 18)


## 7. How each model is trained and evaluated

1. **Tuning:** `GridSearchCV` tries every hyperparameter combination in a small grid with 5-fold cross-validation on the **training set only**, and keeps the best one.
2. **Stability and overfitting:** for the best setting we report the cross-validation accuracy as mean ± standard deviation over the 5 folds, next to its accuracy on the training folds.
   - A small standard deviation means the score does not depend on a lucky split.
   - A training accuracy far above the cross-validation accuracy means the model memorizes the training data (overfitting).
3. **Confusion matrix:** rows are the actual track and columns are the predicted track. The diagonal counts correct predictions; everything off the diagonal is a mistake.
4. **Test-set scores:**
   - **Accuracy:** share of all test responses predicted correctly.
   - **Precision:** of the responses predicted as a track, how many really belong to it.
   - **Recall:** of the responses that really belong to a track, how many the model found.
   - **F1-score:** harmonic mean of precision and recall.

   With 7 tracks, precision, recall and F1 are computed for each track and then averaged with equal weight (**macro average**). The per-track report is printed below the scores.

In [11]:
results = {}


def evaluate(search):
    best = search.best_index_
    scores = {
        "CV accuracy": search.cv_results_["mean_test_score"][best],
        "CV std": search.cv_results_["std_test_score"][best],
        "Train accuracy": search.cv_results_["mean_train_score"][best],
    }
    print("Best parameters:", search.best_params_)
    print(f"Cross-validation accuracy: {scores['CV accuracy']:.4f} ± {scores['CV std']:.4f}")
    print(f"Accuracy on training folds: {scores['Train accuracy']:.4f}")

    y_pred = search.predict(X_test)
    cm = pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=search.classes_),
        index=pd.Index(search.classes_, name="Actual"),
        columns=pd.Index(search.classes_, name="Predicted"),
    )
    print("\nConfusion matrix (test set):")
    display(cm)

    scores["Test accuracy"] = accuracy_score(y_test, y_pred)
    scores["Precision"] = precision_score(y_test, y_pred, average="macro")
    scores["Recall"] = recall_score(y_test, y_pred, average="macro")
    scores["F1-score"] = f1_score(y_test, y_pred, average="macro")
    for metric in ["Test accuracy", "Precision", "Recall", "F1-score"]:
        print(f"{metric:<14} {scores[metric]:.4f}")

    print("\nPer-track report (test set):")
    print(classification_report(y_test, y_pred, digits=4))
    return scores

## 8. Logistic Regression

The features are standardized first (mean 0, standard deviation 1) so the regularization treats every question equally. `C` is the inverse of the regularization strength: smaller values mean stronger regularization and a simpler model.

In [12]:
log_reg = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000))]),
    param_grid={"model__C": [0.001, 0.01, 0.1, 1, 10]},
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
log_reg.fit(X_train, y_train)

results["Logistic Regression"] = evaluate(log_reg)

Best parameters: {'model__C': 0.01}
Cross-validation accuracy: 0.9374 ± 0.0069
Accuracy on training folds: 0.9382

Confusion matrix (test set):


Predicted,Back-end,Cyber Security,Data Science,Front-end,Machine Learning,Mobile applications,UI/UX
Actual,,,,,,,
Back-end,63,0,1,0,0,11,0
Cyber Security,1,60,0,0,0,0,0
Data Science,1,0,62,0,0,0,0
Front-end,0,0,1,59,0,0,1
Machine Learning,0,0,3,0,60,0,0
Mobile applications,3,0,0,2,0,65,0
UI/UX,0,0,0,0,0,0,66


Test accuracy  0.9477
Precision      0.9513
Recall         0.9508
F1-score       0.9503

Per-track report (test set):
                     precision    recall  f1-score   support

           Back-end     0.9265    0.8400    0.8811        75
     Cyber Security     1.0000    0.9836    0.9917        61
       Data Science     0.9254    0.9841    0.9538        63
          Front-end     0.9672    0.9672    0.9672        61
   Machine Learning     1.0000    0.9524    0.9756        63
Mobile applications     0.8553    0.9286    0.8904        70
              UI/UX     0.9851    1.0000    0.9925        66

           accuracy                         0.9477       459
          macro avg     0.9513    0.9508    0.9503       459
       weighted avg     0.9492    0.9477    0.9476       459



## 9. Support Vector Machine (SVM)

We try a linear kernel and an RBF kernel on standardized features. `C` trades a wider margin against training mistakes, and `gamma` controls how far the influence of a single training point reaches with the RBF kernel.

In [13]:
svm = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", SVC())]),
    param_grid=[
        {"model__kernel": ["linear"], "model__C": [0.01, 0.1, 1, 10]},
        {"model__kernel": ["rbf"], "model__C": [0.1, 1, 10, 100], "model__gamma": ["scale", 0.001, 0.01, 0.1]},
    ],
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
svm.fit(X_train, y_train)

results["SVM"] = evaluate(svm)

Best parameters: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Cross-validation accuracy: 0.9390 ± 0.0066
Accuracy on training folds: 0.9517

Confusion matrix (test set):


Predicted,Back-end,Cyber Security,Data Science,Front-end,Machine Learning,Mobile applications,UI/UX
Actual,,,,,,,
Back-end,63,0,0,0,1,11,0
Cyber Security,2,59,0,0,0,0,0
Data Science,2,0,61,0,0,0,0
Front-end,0,0,1,59,0,0,1
Machine Learning,0,0,1,0,62,0,0
Mobile applications,2,0,0,0,0,68,0
UI/UX,0,0,0,0,0,0,66


Test accuracy  0.9542
Precision      0.9588
Recall         0.9569
F1-score       0.9570

Per-track report (test set):
                     precision    recall  f1-score   support

           Back-end     0.9130    0.8400    0.8750        75
     Cyber Security     1.0000    0.9672    0.9833        61
       Data Science     0.9683    0.9683    0.9683        63
          Front-end     1.0000    0.9672    0.9833        61
   Machine Learning     0.9841    0.9841    0.9841        63
Mobile applications     0.8608    0.9714    0.9128        70
              UI/UX     0.9851    1.0000    0.9925        66

           accuracy                         0.9542       459
          macro avg     0.9588    0.9569    0.9570       459
       weighted avg     0.9559    0.9542    0.9542       459



## 10. Random Forest

Trees split on one feature at a time, so they do not need scaled features. Limiting the tree depth and the minimum number of samples in a leaf keeps the trees from memorizing the training data; `max_features` sets how many questions each split may consider.

In [14]:
random_forest = GridSearchCV(
    RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [3, 5, 8, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.5],
    },
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
random_forest.fit(X_train, y_train)

results["Random Forest"] = evaluate(random_forest)

Best parameters: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1}
Cross-validation accuracy: 0.9417 ± 0.0070
Accuracy on training folds: 0.9530

Confusion matrix (test set):


Predicted,Back-end,Cyber Security,Data Science,Front-end,Machine Learning,Mobile applications,UI/UX
Actual,,,,,,,
Back-end,63,0,0,0,1,11,0
Cyber Security,1,60,0,0,0,0,0
Data Science,1,0,62,0,0,0,0
Front-end,0,0,1,59,0,0,1
Machine Learning,0,0,1,0,62,0,0
Mobile applications,0,0,0,2,0,68,0
UI/UX,0,0,0,0,0,0,66


Test accuracy  0.9586
Precision      0.9622
Recall         0.9615
F1-score       0.9607

Per-track report (test set):
                     precision    recall  f1-score   support

           Back-end     0.9692    0.8400    0.9000        75
     Cyber Security     1.0000    0.9836    0.9917        61
       Data Science     0.9688    0.9841    0.9764        63
          Front-end     0.9672    0.9672    0.9672        61
   Machine Learning     0.9841    0.9841    0.9841        63
Mobile applications     0.8608    0.9714    0.9128        70
              UI/UX     0.9851    1.0000    0.9925        66

           accuracy                         0.9586       459
          macro avg     0.9622    0.9615    0.9607       459
       weighted avg     0.9608    0.9586    0.9584       459



## 11. K-Nearest Neighbors (KNN)

KNN predicts the most common track among the closest training responses. The features are standardized so that no single question dominates the distance. We tune the number of neighbors, the distance metric and whether closer neighbors count more.

In [15]:
knn = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    param_grid={
        "model__n_neighbors": [3, 5, 9, 15, 21, 31, 45],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    },
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
knn.fit(X_train, y_train)

results["KNN"] = evaluate(knn)

Best parameters: {'model__metric': 'manhattan', 'model__n_neighbors': 21, 'model__weights': 'uniform'}
Cross-validation accuracy: 0.9357 ± 0.0099
Accuracy on training folds: 0.9360

Confusion matrix (test set):


Predicted,Back-end,Cyber Security,Data Science,Front-end,Machine Learning,Mobile applications,UI/UX
Actual,,,,,,,
Back-end,61,0,0,2,1,11,0
Cyber Security,1,60,0,0,0,0,0
Data Science,0,0,62,0,0,1,0
Front-end,0,0,1,59,0,0,1
Machine Learning,0,0,2,0,61,0,0
Mobile applications,4,0,0,2,0,64,0
UI/UX,0,0,0,0,0,0,66


Test accuracy  0.9434
Precision      0.9465
Recall         0.9473
F1-score       0.9461

Per-track report (test set):
                     precision    recall  f1-score   support

           Back-end     0.9242    0.8133    0.8652        75
     Cyber Security     1.0000    0.9836    0.9917        61
       Data Science     0.9538    0.9841    0.9688        63
          Front-end     0.9365    0.9672    0.9516        61
   Machine Learning     0.9839    0.9683    0.9760        63
Mobile applications     0.8421    0.9143    0.8767        70
              UI/UX     0.9851    1.0000    0.9925        66

           accuracy                         0.9434       459
          macro avg     0.9465    0.9473    0.9461       459
       weighted avg     0.9444    0.9434    0.9430       459



## 12. Compare the models

- **CV accuracy / CV std / Train accuracy** come from cross-validation on the training set.
- **Test accuracy, Precision, Recall and F1-score** are measured on the test set; precision, recall and F1 are macro averages.

We pick the best model by CV accuracy, so the test set plays no part in the choice and its scores stay an honest estimate. Models whose CV accuracy differs by less than about one CV std perform the same for practical purposes.

In [16]:
models = {"Logistic Regression": log_reg, "SVM": svm, "Random Forest": random_forest, "KNN": knn}

summary = pd.DataFrame(results).T.sort_values("CV accuracy", ascending=False)
best_name = summary.index[0]
best_model = models[best_name].best_estimator_
print("Best model by cross-validation accuracy:", best_name)
summary.round(4)

Best model by cross-validation accuracy: Random Forest


,CV accuracy,CV std,Train accuracy,Test accuracy,Precision,Recall,F1-score
Random Forest,0.9417,0.0070,0.9530,0.9586,0.9622,0.9615,0.9607
SVM,0.9390,0.0066,0.9517,0.9542,0.9588,0.9569,0.9570
Logistic Regression,0.9374,0.0069,0.9382,0.9477,0.9513,0.9508,0.9503
KNN,0.9357,0.0099,0.9360,0.9434,0.9465,0.9473,0.9461


## 13. Where do the remaining errors come from?

The first 200 rows of the file look different from the rest: people answered "Yes" to more of the Yes/No questions. They may be the original survey responses. Cross-validation gives every row a prediction from a copy of the best model that did not see that row during training, so we can compare accuracy on these rows with the rest of the file.

In [17]:
predicted = cross_val_predict(best_model, X, y, cv=cv, n_jobs=-1)

errors = pd.DataFrame({
    "Part of the file": np.where(X.index < 200, "First 200 rows", "Rest of the file"),
    "Correct": predicted == y.values,
    "Yes answers": X[question_cols[3:]].sum(axis=1).values,
})
errors.groupby("Part of the file").agg(
    rows=("Correct", "size"),
    accuracy=("Correct", "mean"),
    avg_yes_answers=("Yes answers", "mean"),
).round(3)

,rows,accuracy,avg_yes_answers
Part of the file,,,
First 200 rows,103,0.524,8.680
Rest of the file,2192,0.963,6.083


## 14. Which questions matter most?

**Permutation importance:** shuffle the answers to one question in the test set and measure how much the best model's accuracy drops. A large drop means the model relies on that question. A drop near 0 means the model does not need it: either the question carries no information about the track, or the other answers already give the same information.

In [18]:
importance = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
pd.DataFrame(
    {"Question": question_text, "Accuracy drop": importance.importances_mean},
    index=question_cols,
).sort_values("Accuracy drop", ascending=False).round(3)

,Question,Accuracy drop
Q4,4- Were you willing to learn and work with programming languages?,0.152
Q10,10- Were you interested in data research and analysis?,0.124
Q15,15- Was you choice based on using mobile apps frequently and having ideas for new ones?,0.104
Q13,13- Were you interested in server-side programming and building the backbone of applications?,0.070
Q6,6- Did you have interest creating responsive and interactive web pages,0.058
Q14,14- Did you have interest in creating innovative and creative software?,0.032
Q9,9-Did you enjoy working with data to solve real world problems?,0.012
Q12,12- Did you have interest in using algorithms that can learn and improve from data?,0.007
Q11,11- Were you passionate about data analysis and deriving insights from data sets?,0.005
Q5,"5- Was your choice based on enjoying working with visual elements, such as layout and design?",0.005


## 15. Summary

- **Cleaning the data made the biggest difference.** Two blocks with 1,032 rows in total had random answers: every question was answered "Yes" about half the time, and cross-validated predictions for them were right only about 13% of the time, no better than random guessing (14%). Another 5,089 rows were copies. After removing both, 2,295 unique responses are left.
- **All four models score 94–96% on the test set.** Their cross-validation accuracies (93.6–94.2%) are within about one standard deviation of each other, so the choice of model matters much less than the data cleaning. Random Forest has the highest cross-validation accuracy and the best test scores (accuracy 95.9%, F1-score 96.1%).
- **Little overfitting:** for every model, accuracy on the training folds is at most 1.3 points above its cross-validation accuracy.
- **Most common mistake:** every model predicts 11 of the 75 Back-end test responses as Mobile applications.
- **Weak spot:** accuracy is 96% on most of the file but only about 52% on the first 200 rows, where people answered "Yes" to more questions. If those rows are the real survey responses, expect lower accuracy on new real respondents than the test score suggests.
- **Most useful questions:** Q4 (willing to learn programming languages), Q10 (data research), Q15 (mobile apps), Q13 (server-side programming) and Q6 (web pages). The model does not use Q1–Q3 (programming level, and how often people code or follow tech news).